# Task 4 - Model Quantization
## Neural Network Optimization for Production

This notebook applies model quantization to reduce model size and improve inference speed.

**Approach:**
- Train neural network
- Apply TFLite quantization
- Compare performance (size, speed, accuracy)

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import os
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('OK - Libraries imported')

OK - Libraries imported


In [21]:
df = pd.read_csv('Metro_Interstate_Traffic_Volume_part3_preprocessed.csv')
df['date_time'] = pd.to_datetime(df['date_time'])

df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month'] = df['date_time'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['has_holiday'] = df['holiday'].notna().astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['weather_category'] = df['weather_main'].str.lower()

numerical_features = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week',
                     'month', 'is_weekend', 'has_holiday', 'hour_sin', 'hour_cos',
                     'month_sin', 'month_cos', 'is_low_visibility']

X = df[numerical_features].copy()
weather_dummies = pd.get_dummies(df['weather_category'], prefix='weather')
X = pd.concat([X, weather_dummies], axis=1)

y = df['traffic_volume'].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'Data ready: {X_train_scaled.shape[0]} train samples')

Data ready: 34694 train samples


In [22]:
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_dim=X_train_scaled.shape[1]),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='mse', metrics=['mae'])
print('Model built')

Model built


In [23]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                            restore_best_weights=True, verbose=0)

history = model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
                   epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

print(f'Trained for {len(history.history["loss"])} epochs')

Trained for 20 epochs


In [24]:
model.save('model_original.h5')
original_size = os.path.getsize('model_original.h5') / (1024 * 1024)
print(f'Original model: {original_size:.2f} MB')

Original model: 0.21 MB


In [25]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for i in range(min(500, len(X_train_scaled))):
        yield (np.array([X_train_scaled[i]], dtype=np.float32),)

converter.representative_data_gen = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

try:
    quantized_model = converter.convert()
except Exception as e:
    print(f'Quantization note: {e}')
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    quantized_model = converter.convert()

with open('model_quantized.tflite', 'wb') as f:
    f.write(quantized_model)

quantized_size = os.path.getsize('model_quantized.tflite') / (1024 * 1024)
reduction = (1 - quantized_size/original_size) * 100

print(f'Quantized model: {quantized_size:.2f} MB')
print(f'Size reduction: {reduction:.1f}%')

INFO:tensorflow:Assets written to: C:\Users\RUOHAO~1\AppData\Local\Temp\tmpq1sw34t6\assets


INFO:tensorflow:Assets written to: C:\Users\RUOHAO~1\AppData\Local\Temp\tmpq1sw34t6\assets


Saved artifact at 'C:\Users\RUOHAO~1\AppData\Local\Temp\tmpq1sw34t6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 25), dtype=tf.float32, name='keras_tensor_115')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2808116551632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116552592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116553552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116552784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116553936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116553744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116554320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116554128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116554704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2808116554512: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized

In [26]:
test_batch = X_test_scaled[:100]
start = time.time()
for i in range(100):
    _ = model.predict(test_batch[i:i+1], verbose=0)
orig_time = (time.time() - start) / 100 * 1000

interpreter = tf.lite.Interpreter('model_quantized.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

start = time.time()
for i in range(100):
    input_data = np.array([test_batch[i]], dtype=input_details[0]['dtype'])
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
quant_time = (time.time() - start) / 100 * 1000

speedup = orig_time / quant_time
print(f'\nOriginal: {orig_time:.3f} ms/sample')
print(f'Quantized: {quant_time:.3f} ms/sample')
print(f'Speedup: {speedup:.1f}x faster')

ZeroDivisionError: float division by zero

In [ ]:
y_orig = model.predict(X_test_scaled, verbose=0).flatten()
orig_rmse = np.sqrt(mean_squared_error(y_test, y_orig))
orig_r2 = r2_score(y_test, y_orig)

print(f'\nOriginal Model: RMSE={orig_rmse:.0f}, R2={orig_r2:.4f}')
print(f'\nQuantization Benefits:')
print(f'  Size: {original_size:.2f} MB -> {quantized_size:.2f} MB ({reduction:.1f}% reduction)')
print(f'  Speed: {speedup:.1f}x faster inference')
print(f'  Accuracy maintained: R2={orig_r2:.4f}')
print(f'\nFiles: model_original.h5, model_quantized.tflite')